## Introduction to the Data:

**Source** (*Where to download data*): https://www.trainingproviderresults.gov/#!/about
**File:** `DownloadPrograms.csv`
**Published by:** U.S. Department of Labor

### What it is
A list of job training programs that people can pay for with federal WIOA training money. States report how each program's students did: how many finished, got jobs, and how much they earned. Think of it as the "College Scorecard" for job training programs.

**Each row = one program at one school or training provider.**
Size: 77,085 rows × 57 columns.

### What's in it
- **Provider info:** name, type, location
- **Program info:** name, credential, field of study (CIP code), format (online/in-person)
- **Cost and length:** tuition, supplies, weeks
- **Outcomes:** students served, completed, employed, median earnings

### Why we're using it
Scorecard and IPEDS mostly cover colleges. This data also covers short-term certificates and training programs, which are the programs Workforce Pell is meant to fund.

### Things to watch for
- Small programs have hidden outcomes (shown as `-1`)
- Earnings are per quarter, not per year
- No college ID, so we have to match on school name and location
- Some codes got messed up by Excel

--------

## Setup & Load the Data

In [122]:
import pandas as pd
import numpy as np

RAW_PATH = "../../../data/raw/ETPL/DownloadPrograms.csv"

# encoding="utf-8-sig" strips the invisible BOM character at the start of the file,
# otherwise the first column name becomes "﻿d101_eligible_training_provider"
# low_memory=False makes pandas read the whole file before guessing column types

raw = pd.read_csv(RAW_PATH, encoding="utf-8-sig", low_memory=False)

# Work on a copy so the raw data stays untouched for comparisons later
df = raw.copy()

print("Shape:", df.shape)
df.head(3)

Shape: (77085, 57)


,d101_eligible_training_provider,d103_provider_address,d104_entity_type,d105_program_name,d106_program_description,d107_program_url,d108_program,d109_associated_credential,d110_cip_code,d111_non_wioa_tuition_cost,...,long,cip_formatted_4,reportingstate,CIP_Title,provider_unique_id,de129,de130,de170,de171,de172
0,Northwest Shoals Community College,800 George Wallace Blvd. P.O. Box 2 Muscle Sho...,Higher Ed: Associate's Degree,Diagnostic Medical Sonography,A program that prepares individuals under the...,https://www.nwscc.edu,134,Associate in Applied Science Degree,51.0910,8160.00,...,-87.64,51.09,AL,"Allied Health Diagnostic, Intervention, and Tr...",54,12,12,-1,-1,7/26/22
1,"DLS Medical Billing and Coding, LLC","1029B W Fort Williams St Sylacauga, AL 35150",Public,DLS Medical Billing and Coding Certification P...,A program that prepares individuals to perform...,http://www.dlsmed.com,1,Medical Billing and Coding Certification,51.0714,2400.00,...,-86.26,51.07,AL,Health and Medical Administrative Services.,24,24,16,-1,-1,11/22/23
2,George C. Wallace Community College,"1141 Wallace Drive Dothan, AL 36303",Higher Ed: Associate's Degree,Electrical Technology AAS,The Electrical Technology associate degree pro...,https://www.wallace.edu/programs_of_study/care...,148,Associate in Applied Science in Electrical Tec...,46.0302,11899.45,...,-85.40,46.03,AL,Electrical and Power Transmission Installers.,34,69,59,3,3,12/13/23


**Notes:**

- The data loaded correctly: **77,085 rows × 57 columns**, which matches the raw file.
- Each row is one program at one provider.
- Column names start with codes like `d101`, `d110`, etc. These match the field numbers in the ETP Data Dictionary.
- Already visible in the first 3 rows:
  - `d104_entity_type` isn't always reliable. "DLS Medical Billing and Coding, LLC" is listed as **Public**, even though it's a private company.
  - `d108_program` has numbers (134, 1, 148) that don't mean anything for our analysis.
  - Descriptions and URLs are long text and won't be used in modeling.
- We kept an untouched copy (`raw`) so we can compare before vs. after cleaning.

--------------

## Turn `-1` placeholders into real missing values (NaN)

In [123]:
# Every count, rate, earnings, and length column uses -1 to mean "not reported / suppressed".
# If we leave -1 in, averages and ratios are wrong (e.g. -1 / -1 = a "100%" completion rate).

# de130 was read in as text, so convert it to a number first
df["de130"] = pd.to_numeric(df["de130"], errors="coerce")

# All numeric columns where -1 is a placeholder
neg1_cols = [
    "d113_program_length_hours", "d114_program_length_weeks",
    "d120_total_served", "d121_total_exited", "d122_total_completed",
    "d123_total_employed_q2", "d124_total_employed_q4", "d125_median_earnings",
    "d126_total_credential",
    "d133_total_wioa_served", "d134_total_wioa_exiters",
    "d135_total_wioa_served_with_ita", "d136_total_wioa_exited_with_ita",
    "d137_total_wioa_completed", "d139_total_wioa_exiters_employed_q2",
    "d140_total_wioa_exiters_employed_q4", "d142_total_wioa_credential",
    "c_wioa_completed_percent", "c_total_employed_WIOA_q2_percent",
    "c_total_employed_WIOA_q4_percent", "c_completed_percent",
    "c_wioa_earned_cred_percent", "c_cost_per_wioa", "c_q2_employment_percent",
    "de129", "de130", "de170", "de171",
]

before = (df[neg1_cols] == -1).sum()
df[neg1_cols] = df[neg1_cols].replace(-1, np.nan)

# Cost per WIOA participant = 0 in ~53k rows. That means "no WIOA participants", not free training.
df["d138_cost_per_wioa_num"] = df["d138_cost_per_wioa_num"].replace(0, np.nan)

# Flag: does this program report ANY outcome data?
# Small programs are suppressed (served starts at 10, other counts at 3), so missing outcomes are
# NOT random. Keep this flag so we can later compare programs with vs. without outcomes.
outcome_cols = ["d122_total_completed", "d123_total_employed_q2", "d125_median_earnings"]
df["has_outcomes"] = df[outcome_cols].notna().any(axis=1)

print("-1 values replaced per column:")
print(before[before > 0].sort_values(ascending=False).to_string())
print("\nPrograms with any outcome data:", df["has_outcomes"].sum(), "of", len(df))

-1 values replaced per column:
c_wioa_earned_cred_percent             70180
d142_total_wioa_credential             70180
d136_total_wioa_exited_with_ita        69133
c_total_employed_WIOA_q4_percent       68545
d140_total_wioa_exiters_employed_q4    68545
d139_total_wioa_exiters_employed_q2    68267
c_total_employed_WIOA_q2_percent       68267
c_wioa_completed_percent               68084
d137_total_wioa_completed              68084
c_cost_per_wioa                        68061
d135_total_wioa_served_with_ita        67822
de171                                  67343
de170                                  67175
d134_total_wioa_exiters                66391
d133_total_wioa_served                 64716
d126_total_credential                  49091
d125_median_earnings                   48410
d124_total_employed_q4                 47824
c_q2_employment_percent                47096
d123_total_employed_q2                 47016
de130                                  44372
de129                   

**Notes:**

- Replaced every `-1` with a blank (NaN) so it won't throw off averages or rates.
- **WIOA-only columns (`d133`–`d142`) are about 85–91% blank.** Most programs had too few WIOA-funded students to report, so we'll mostly use the all-student columns instead.
- **All-student outcomes are about 55–64% blank:**
  - Completed (`d122`): 43,062 blank
  - Employed Q2 (`d123`): 47,016 blank
  - Median earnings (`d125`): 48,410 blank
- **Program length is blank for 19,228 programs (25%).** Hours and weeks are always blank together.
- **Only 35,611 of 77,085 programs (46%) have any outcome data.** The rest are mostly small programs whose results were hidden for privacy.
- This missing data is **not random**: smaller programs are the ones missing. We added a `has_outcomes` flag so we can later check whether programs with data look different from programs without it.
- Also changed `d138_cost_per_wioa_num` from 0 to blank. A 0 there means "no WIOA students," not "free."

## Drop columns and rename unlabeled ones

In [124]:
# c_total_emp_q2_perc_comp -> 100% empty
# d108_program             -> internal code with no clear meaning (max = 123456890)
# d106 / d107 (description, URL) -> free text; saved to a separate lookup file, not used for modeling
lookup_cols = ["provider_unique_id", "d105_program_name", "d106_program_description", "d107_program_url"]
program_text_lookup = df[lookup_cols].copy()

drop_cols = ["c_total_emp_q2_perc_comp", "d108_program",
             "d106_program_description", "d107_program_url"]
df = df.drop(columns=drop_cols)

# de129 is the denominator of c_q2_employment_percent (employed_q2 / de129 matches 100% of rows),
# so it's the number of exiters in the Q2 employment cohort. Give it a readable name.
# de130 / de170 / de171 are still UNCONFIRMED -> check the TrainingProviderResults.gov data
# dictionary before using them. Their names are prefixed "unconfirmed_" as a reminder.
df = df.rename(columns={
    "de129": "q2_cohort_exiters",
    "de130": "unconfirmed_de130",
    "de170": "unconfirmed_de170",
    "de171": "unconfirmed_de171",
    "de172": "last_updated",
})

print("Shape after dropping:", df.shape)

Shape after dropping: (77085, 54)


## Rebuild CIP and SOC codes (w/ the merge keys)

In [125]:
# --- CIP ---
# d110_cip_code was saved as a number, so zeros got lost:
#   51.0910 -> 51.091   and   01.0000 -> 1.0
# Format back to the standard 6-digit CIP text "XX.XXXX" (7 characters including the dot).
df["cip6"] = df["d110_cip_code"].map(lambda x: f"{x:07.4f}" if pd.notna(x) else np.nan)

# 4-digit CIP for joining with Scorecard / PSEO (built from cip6, NOT from cip_formatted_4,
# which chops digits off and disagrees with the full code in ~4k rows)
df["cip4"] = df["cip6"].str[:5]                         # "51.09" (with dot)
df["cip4_nodot"] = df["cip4"].str.replace(".", "", regex=False)   # "5109" (Scorecard style)
df["cip2"] = df["cip6"].str[:2]                         # broad field, e.g. "51" = Health

df = df.drop(columns=["d110_cip_code", "cip_formatted_4"])

# --- SOC ---
# SOC codes come in 3 shapes:
#   "29-203200" (9 chars) -> O*NET-style with 2 extra digits -> keep first 6 digits
#   "25-1081"   (7 chars) -> already standard
#   "Nov-41"    (6 chars) -> Excel turned "11-41xx" into a date; the detail is lost -> set NaN
def clean_soc(code):
    if pd.isna(code):
        return np.nan
    digits = "".join(ch for ch in str(code) if ch.isdigit())
    if len(digits) < 6:          # corrupted / incomplete (e.g. the "Nov-41" date bug)
        return np.nan
    return f"{digits[:2]}-{digits[2:6]}"   # standard "XX-XXXX"

soc_cols = ["d117_program_soc_occupation_1", "d118_program_soc_occupation_2", "d119_program_soc_occupation_3"]
for i, col in enumerate(soc_cols, start=1):
    bad_before = df[col].notna().sum()
    df[f"soc_{i}"] = df[col].map(clean_soc)
    print(f"soc_{i}: {bad_before - df[f'soc_{i}'].notna().sum()} codes unrecoverable -> NaN")
df = df.drop(columns=soc_cols)

print("\nExample codes:")
print(df[["cip6", "cip4", "cip4_nodot", "soc_1"]].head())

soc_1: 1741 codes unrecoverable -> NaN
soc_2: 259 codes unrecoverable -> NaN
soc_3: 404 codes unrecoverable -> NaN

Example codes:
      cip6   cip4 cip4_nodot    soc_1
0  51.0910  51.09       5109  29-2032
1  51.0714  51.07       5107  13-1151
2  46.0302  46.03       4603  47-2111
3  51.0904  51.09       5109  29-2042
4  51.0801  51.08       5108  31-9092


**Notes:**

- **CIP codes are fixed.** Codes that had lost zeros are back in the standard 6-digit form (e.g., `51.091` → `51.0910`).
- We built 3 versions of the CIP code for merging:
  - `cip6` (e.g., `51.0910`): the full 6-digit code
  - `cip4` (e.g., `51.09`): the 4-digit field of study
  - `cip4_nodot` (e.g., `5109`): the same code in College Scorecard's format, for joining
- Example: `51.0910` = Diagnostic Medical Sonography, which falls under `51.09` (Allied Health Diagnostic Professions).
- **SOC codes are now all in the standard `XX-XXXX` format** (e.g., `29-203200` → `29-2032`).
- **Some SOC codes couldn't be saved:** 1,741 in `soc_1` (about 2%), 259 in `soc_2`, and 404 in `soc_3`. Excel had turned these into dates (e.g., "Nov-41"), so the original code is gone. They're now blank.
- These codes are the **main keys for merging** with Scorecard, PSEO, IPEDS, and BLS, so getting them right matters more than any other step.

## Categorize credentials, entity type, and format (scope flags)

In [126]:
import re

# The credential column is free text with ~22k distinct spellings. Group into a few categories.
# ORDER MATTERS: more specific patterns are checked first (e.g. "post-bacc certificate" before "certificate").
def credential_category(text):
    if pd.isna(text):
        return "missing"
    t = str(text).lower()
    if "no credential" in t:
        return "none"
    if "post-bacc" in t or "post-bachelor" in t or "master" in t or "graduate" in t:
        return "graduate"
    # Degree abbreviations at the start, e.g. "BA-4 Year Degree", "BAS", "BSN", "AS-2 YEAR DEGREE", "AAS-T"
    if "bachelor" in t or "baccalaureate" in t or "bachalaureate" in t or re.match(r"^(ba|bs|bas|bsn|ba/bs)\b", t):
        return "bachelors"
    if "associate" in t or re.match(r"^(aa|as|aas|aat|aas-t|aos)\b", t):
        return "associate"
    if "apprentice" in t or "journey" in t:
        return "apprenticeship"
    if "license" in t or "licensure" in t or "registered nurse" in t:
        return "license"
    # Named industry credentials: "Certified Nursing Assistant", "CompTIA A+", "CDL", "NCCER", ...
    if ("industry" in t or "certification" in t or t.startswith("certified")
            or re.search(r"\b(comptia|cdl|nccer|osha|aws|cisco|microsoft)\b", t)):
        return "industry_certification"
    if "certificate" in t or "sub-bachelor" in t or "diploma" in t or t.startswith("c0") or t.startswith("c1"):
        return "certificate"
    return "other"

df["credential_category"] = df["d109_associated_credential"].map(credential_category)

# Shorten the long format sentences into simple labels
def format_label(text):
    t = str(text).lower()
    if "multiple formats" in t:
        return "multiple"
    if "hybrid" in t:
        return "hybrid"
    if "online" in t:
        return "online"
    if "in-person" in t:
        return "in_person"
    return np.nan

df["program_format"] = df["d116_program_format"].map(format_label)

# Flags for scope decisions (we FLAG instead of deleting, so the team can decide later)
higher_ed = ["Higher Ed: Associate's Degree", "Higher Ed: Baccalaureate or Higher",
             "Higher Ed: Certificate of Completion"]
df["is_higher_ed_provider"] = df["d104_entity_type"].isin(higher_ed)   # most likely to match IPEDS/Scorecard
df["is_undergrad_certificate"] = df["credential_category"].isin(["certificate", "industry_certification"])

print(df["credential_category"].value_counts().to_string())
print()
print(df["program_format"].value_counts(dropna=False).to_string())

# Spot-check what fell into "other" so you can add rules if needed
print("\nTop 'other' credential texts:")
print(df.loc[df["credential_category"] == "other", "d109_associated_credential"].value_counts().head(15).to_string())

credential_category
certificate               19185
industry_certification    14105
associate                 13007
other                     11482
bachelors                  6874
missing                    6017
license                    2828
none                       1561
apprenticeship             1476
graduate                    550

program_format
in_person    40573
hybrid       20698
online       13058
multiple      2756

Top 'other' credential texts:
d109_associated_credential
City College of San Francisco            176
Credential                               116
Pharmacy Technician                       70
Comp TIA A+                               63
Phlebotomy Technician                     59
ATA                                       55
Lanier Technical College                  54
Skills Progression                        50
Medical Assistant                         50
Degree                                    48
Dental Assistant                          47
CNA            

## Make units consistent (earnings, length, rates)

In [127]:
# --- Earnings ---
# d125_median_earnings is median earnings for ONE QUARTER after exit (median ~ $8.6k, max ~ $25k).
# Scorecard / PSEO report ANNUAL earnings, so multiply by 4 to compare.
df = df.rename(columns={"d125_median_earnings": "median_earnings_quarterly"})
df["median_earnings_annual"] = df["median_earnings_quarterly"] * 4

# --- Program length ---
# d113 tops out at exactly 50 with spikes at 40/50 -> looks like HOURS PER WEEK, not total hours.
# ASSUMPTION (verify in data dictionary): total contact hours = hours/week * weeks.
df = df.rename(columns={"d113_program_length_hours": "hours_per_week",
                        "d114_program_length_weeks": "length_weeks"})
df["total_contact_hours"] = df["hours_per_week"] * df["length_weeks"]

# --- Rates: recompute from the counts instead of trusting the c_* columns ---
# Verified in the data: completion rate = completed / EXITED (not served),
#                       Q2 employment rate = employed_q2 / q2_cohort_exiters (old de129)
def safe_rate(num, den):
    # NaN when either side is missing or denominator is 0
    return np.where((den > 0) & num.notna(), num / den, np.nan)

df["completion_rate"]   = safe_rate(df["d122_total_completed"],   df["d121_total_exited"])
df["employment_q2_rate"] = safe_rate(df["d123_total_employed_q2"], df["q2_cohort_exiters"])
df["credential_rate"]   = safe_rate(df["d126_total_credential"],  df["d121_total_exited"])

# Drop the provided c_* rate columns now that we've rebuilt the main ones
c_cols = [c for c in df.columns if c.startswith("c_")]
df = df.drop(columns=c_cols)

print(df[["median_earnings_annual", "total_contact_hours",
          "completion_rate", "employment_q2_rate", "credential_rate"]].describe().round(2))

       median_earnings_annual  total_contact_hours  completion_rate  \
count                28675.00             57857.00         34022.00   
mean                 36892.66              1416.38             0.68   
std                  16149.33              2195.23             0.27   
min                     12.00                 1.00             0.00   
25%                  25846.16               192.00             0.47   
50%                  34375.44               640.00             0.73   
75%                  44987.20              1560.00             0.93   
max                  99939.00             13000.00             1.00   

       employment_q2_rate  credential_rate  
count            30069.00         27930.00  
mean                 0.70             0.55  
std                  0.21             0.38  
min                  0.00             0.00  
25%                  0.59             0.29  
50%                  0.73             0.50  
75%                  0.84             0.74  


**Notes:**

- **Earnings converted to yearly** (quarterly × 4) so they can be compared with Scorecard and PSEO.
  - Median annual earnings: **~$34,400**. The middle half of programs falls between **~$25,800 and ~$45,000**.
  - The lowest value is **$12/year**, which can't be real. Step 6 handles it.
- **Total contact hours** (hours per week × weeks):
  - Median: **640 hours**, about 16 weeks at 40 hrs/week
  - The middle half of programs falls between **192 and 1,560 hours**
  - The max of 13,000 hours (50 hrs/week × 260 weeks) is probably a 4–5 year degree program
  - This assumes `d113` means hours **per week**. Still need to confirm this in the data dictionary.
- **Rates recalculated from the counts** instead of using the file's own percentages:
  - Completion rate (completed ÷ exited): median **73%**
  - Q2 employment rate (employed ÷ Q2 cohort): median **73%**
  - Credential rate (credentials ÷ exited): median **50%**
- **The credential rate goes as high as 10.25 (1,025%)**, which is impossible. Some programs report more credentials than students, either because students earned several credentials each or because the counts come from different years. Step 6 caps these at 100%.
- Only about **28k–34k programs** have each rate, so models built on these will use a smaller set of programs.

---

## Logical consistency checks and outliers

In [128]:
# Rates above 1 are impossible (e.g. more people completed than exited). This usually happens
# because the counts come from different reporting cohorts/time windows. FLAG them, then cap at 1.
rate_cols = ["completion_rate", "employment_q2_rate", "credential_rate"]
df["flag_rate_over_1"] = (df[rate_cols] > 1).any(axis=1)
for col in rate_cols:
    df[col] = df[col].clip(upper=1)

# Other impossible combos worth flagging
df["flag_completed_gt_served"] = df["d122_total_completed"] > df["d120_total_served"]

# --- Tuition outliers ---
# Median ~$4.2k but max ~$450k. Flag anything above the 99.5th percentile and make a capped
# ("winsorized") copy for modeling. The original column stays for manual review.
tuition_cap = df["d111_non_wioa_tuition_cost"].quantile(0.995)
df["flag_tuition_outlier"] = df["d111_non_wioa_tuition_cost"] > tuition_cap
df["tuition_capped"] = df["d111_non_wioa_tuition_cost"].clip(upper=tuition_cap)

# --- Implausibly low earnings ---
# Some programs report annual median earnings of only a few dollars (min = $12/yr). These are
# almost certainly reporting errors. Flag anything under $1,000/yr (~1 week of full-time minimum
# wage per quarter) and set it to NaN so it doesn't drag down models.
df["flag_low_earnings"] = df["median_earnings_annual"] < 1000
df.loc[df["flag_low_earnings"], ["median_earnings_annual", "median_earnings_quarterly"]] = np.nan
print("Implausibly low earnings set to NaN:", df["flag_low_earnings"].sum())

print(f"Tuition cap (99.5th pct): ${tuition_cap:,.0f}")
print("Rows with a rate > 1 (now capped):", df["flag_rate_over_1"].sum())
print("Completed > served:", df["flag_completed_gt_served"].sum())
print("Tuition outliers:", df["flag_tuition_outlier"].sum())

# Look at the biggest tuition values by hand
df.loc[df["flag_tuition_outlier"],
       ["d101_eligible_training_provider", "d105_program_name",
        "credential_category", "d111_non_wioa_tuition_cost"]
      ].sort_values("d111_non_wioa_tuition_cost", ascending=False).head(10)

Implausibly low earnings set to NaN: 134
Tuition cap (99.5th pct): $79,600
Rows with a rate > 1 (now capped): 1181
Completed > served: 890
Tuition outliers: 372


,d101_eligible_training_provider,d105_program_name,credential_category,d111_non_wioa_tuition_cost
32023,Metropolitan Community College,Industrial Technologies-Multi-Craft,associate,450241.0
42690,Pennsylvania Western University,Associate of Science in Nursing,associate,333500.0
19481,Rio Grande Valley College,Physical Therapist Assistant,associate,295000.0
29211,"Missouri State University, West Plains",Advanced Welding And Fabrication Technology,certificate,270000.0
37634,Upper Iowa University,Exercise/Sport Studies,bachelors,260712.0
43069,POINT PARK UNIVERSITY,Bachelor of Science in Electrical Engineering,bachelors,214070.0
6232,College of St. Benedict,English,bachelors,210800.0
9670,Saint John's University,Nutrition,bachelors,210800.0
7075,College of St. Benedict,Elementary Education,bachelors,210800.0
6920,Saint John's University,Elementary Education,bachelors,210800.0


**Notes:**

- **134 programs had annual earnings under $1,000** (as low as $12/year). These are almost certainly reporting errors, so we flagged them and set them to blank.
- **1,181 programs had a rate above 100%** (e.g., more credentials than students). We flagged these and capped the rates at 100%. This probably happens because the counts come from different years or groups of students.
- **890 programs report more completers than students served**. These are flagged so we can leave them out if needed.
- **Tuition cap = $79,600** (the 99.5th percentile). **372 programs** are above it.
  - We kept the original tuition and added a capped version (`tuition_capped`) for modeling.

**Looking at the top 10 tuition values:**
- **Most are bachelor's programs reporting the full 4-year cost.** For example, College of St. Benedict / Saint John's at $210,800 works out to ~$52,700 per year, which is realistic for a private university.
- **Some are clearly mistakes:**
  - A community college associate degree at **$450,241** (Metropolitan Community College)
  - A welding **certificate** at **$270,000** (Missouri State, West Plains), possibly a typo for $2,700
- Only 1 of the top 10 is a certificate, so outliers mostly affect degree programs, **not our main focus**.
- Takeaway: tuition isn't reported the same way everywhere (some per year, some for the full program), so compare costs carefully.

## Identifying duplicates and stale records

In [129]:
# last_updated (old de172) is text like "7/26/22" -> convert to a real date
df["last_updated"] = pd.to_datetime(df["last_updated"], format="%m/%d/%y", errors="coerce")
print("Unparseable dates:", df["last_updated"].isna().sum())
print(df["last_updated"].dt.year.value_counts().sort_index().to_string())

# No exact duplicate rows, but the same provider + program + CIP shows up more than once
# (often the same program reported by two different states). Keep the most recently updated one.
dup_key = ["provider_unique_id", "d105_program_name", "cip6"]
n_before = len(df)
df = (df.sort_values("last_updated", ascending=False)
        .drop_duplicates(subset=dup_key, keep="first")
        .sort_index())
print(f"\nDuplicates removed: {n_before - len(df)}  ->  {len(df)} rows left")

# Stale flag: not updated in the last ~3 years (adjust the cutoff as a team)
STALE_CUTOFF = pd.Timestamp("2022-01-01")
df["flag_stale"] = df["last_updated"] < STALE_CUTOFF
print("Stale records (before 2022):", df["flag_stale"].sum())

Unparseable dates: 0
last_updated
2014     3796
2015     1913
2016     1802
2017     1755
2018     1702
2019     5302
2020     4542
2021     8754
2022    12886
2023    17335
2024    17298

Duplicates removed: 3481  ->  73604 rows left
Stale records (before 2022): 26941


## Location cleanup (state, city, ZIP, coordinates)

In [130]:
# --- State ---
# Fix casing ("Oh", "Tx", "Va" -> "OH", "TX", "VA")
df["state"] = df["state"].str.strip().str.upper()
df["reportingstate"] = df["reportingstate"].str.strip().str.upper()

# Flag rows outside the 50 states + DC (territories and "MB" = Manitoba, Canada)
territories = ["PR", "GU", "VI", "MP", "AS"]
df["is_territory"] = df["state"].isin(territories)
df["is_non_us"] = df["state"].isin(["MB"])
print("Territory rows:", df["is_territory"].sum(), "| Non-US rows:", df["is_non_us"].sum())
# NOTE: state != reportingstate is normal (program approved in a neighboring state).
#       Use `state` for location; keep `reportingstate` as its own field.

# --- ZIP ---
# Store as 5-character text so leading zeros survive (603 -> "00603", Puerto Rico)
df["zip"] = (pd.to_numeric(df["zip"].astype(str).str[:5], errors="coerce")
               .map(lambda z: f"{int(z):05d}" if pd.notna(z) else np.nan))

# --- City ---
# "Na" is a placeholder, not a city. Set to NaN, then fill from other rows with the same ZIP.
df["city"] = df["city"].replace({"Na": np.nan, "NA": np.nan}).str.strip()
zip_to_city = (df.dropna(subset=["city", "zip"])
                 .groupby("zip")["city"]
                 .agg(lambda s: s.mode().iloc[0]))
missing_city_before = df["city"].isna().sum()
df["city"] = df["city"].fillna(df["zip"].map(zip_to_city))
print(f"Missing cities: {missing_city_before} -> {df['city'].isna().sum()}")

# --- Coordinates ---
# US mainland longitudes are negative. ~2,100 rows have the sign flipped (e.g. IL at +89).
# Guam / Northern Mariana Islands really are positive (east of the date line), so skip those.
positive_ok = ["GU", "MP"]
flip = (df["long"] > 0) & (~df["state"].isin(positive_ok))
df.loc[flip, "long"] = -df.loc[flip, "long"]
print("Longitudes sign-flipped:", flip.sum())
# Lat/long are rounded to 2 decimals (~1 km); fine for a state/county map, not for street level.

Territory rows: 399 | Non-US rows: 1
Missing cities: 974 -> 144
Longitudes sign-flipped: 2088


## Text cleanup and a provider-name key for matching to IPEDS / Scorecard

In [131]:
import re

# Commas inside fields were replaced by double spaces -> collapse repeated whitespace
text_cols = ["d101_eligible_training_provider", "d103_provider_address", "d105_program_name",
             "d109_associated_credential", "address", "CIP_Title"]
for col in text_cols:
    df[col] = df[col].astype("string").str.replace(r"\s+", " ", regex=True).str.strip()

# This file has NO UNITID/OPEID, so joining to IPEDS/Scorecard needs name + location matching.
# Build a standardized name to make fuzzy matching easier.
ABBREVIATIONS = {
    r"\bcomm\b": "community",
    r"\bcc\b": "community college",
    r"\bcoll\b": "college",
    r"\buniv\b": "university",
    r"\btech\b": "technical",
    r"\binst\b": "institute",
    r"\bst\b": "state",      # careful: may also mean "saint" -- review matches by hand
    r"&": " and ",
}

def clean_name(name):
    if pd.isna(name):
        return np.nan
    n = str(name).lower()
    n = re.sub(r"[^\w\s&]", " ", n)          # drop punctuation (keep & for the rule above)
    for pattern, repl in ABBREVIATIONS.items():
        n = re.sub(pattern, repl, n)
    n = re.sub(r"\b(the|inc|llc|corp)\b", " ", n)   # drop filler words
    return re.sub(r"\s+", " ", n).strip()

df["provider_name_clean"] = df["d101_eligible_training_provider"].map(clean_name)

# Match key = clean name + state (add city/zip for a stricter match)
df["provider_match_key"] = df["provider_name_clean"] + "|" + df["state"]

print(df[["d101_eligible_training_provider", "provider_name_clean"]].drop_duplicates().head(10).to_string())

                 d101_eligible_training_provider                         provider_name_clean
0             Northwest Shoals Community College          northwest shoals community college
1            DLS Medical Billing and Coding, LLC              dls medical billing and coding
2            George C. Wallace Community College          george c wallace community college
3   Wallace State Community College - Hanceville  wallace state community college hanceville
4                 LB Allied Health Training Cent              lb allied health training cent
5                Shelton State Community College             shelton state community college
6                 Bishop State Community College              bishop state community college
7                 Lawson State Community College              lawson state community college
9                Gadsden State Community College             gadsden state community college
11                     Calhoun Community College                   cal

## Flag likely Workforce Pell programs

In [132]:
# Workforce Pell length rules: 150-599 clock hours AND 8 to <15 weeks
# NOTE: assumes total_contact_hours = hours/week x weeks (confirm d113 in the data dictionary)
df["wp_length_ok"] = (
    df["total_contact_hours"].between(150, 599) &
    (df["length_weeks"] >= 8) & (df["length_weeks"] < 15)
)

# Workforce Pell outcome rules: >=70% completion and >=70% job placement
# Q2 employment (2nd quarter after exit, ~180 days) is our closest match to the placement rate
df["wp_outcomes_ok"] = (df["completion_rate"] >= 0.70) & (df["employment_q2_rate"] >= 0.70)

# Likely Workforce Pell = right length + certificate-type credential + higher-ed provider
df["likely_workforce_pell"] = (
    df["wp_length_ok"] &
    df["is_undergrad_certificate"] &
    df["is_higher_ed_provider"]
)

wp = df[df["likely_workforce_pell"]]
print("Programs that meet the length rules:", df["wp_length_ok"].sum())
print("Likely Workforce Pell programs:", len(wp))
print("...that also meet both 70% outcome rules:", wp["wp_outcomes_ok"].sum())

Programs that meet the length rules: 3668
Likely Workforce Pell programs: 912
...that also meet both 70% outcome rules: 81


**Notes:**

- **3,668 programs** fit the Workforce Pell length rules (150–599 hours, 8 to under 15 weeks).
- **912 programs** are "likely Workforce Pell": the right length, a certificate-type credential, and a higher-ed provider.
- **Only 81 of those 912 (9%) meet both 70% rules** (completion and job placement).
  - That number is low partly because **many of the 912 have no outcome data** (hidden for privacy). A blank counts as "not meeting" the rule, so 81 is a minimum, not a final count.
  - It's also low because 70% on **both** measures is a high bar.
- **We kept programs that fail the 70% rules.** Our model needs both strong and weak programs to learn the difference, so `wp_outcomes_ok` will be something we predict, not a filter.
- **What we can't check with this data:** state approval, links to in-demand jobs, whether the program is 1+ year old, and the value-added earnings test.
- **Caveat:** the length rules assume `d113` is hours per week. If the data dictionary says it's total hours, these counts will change.

--------

## Final Review / Saving

In [133]:
# Put the most-used columns first
front = ["provider_unique_id", "d101_eligible_training_provider", "provider_name_clean",
         "provider_match_key", "city", "state", "zip", "d104_entity_type",
         "d105_program_name", "cip6", "cip4", "cip4_nodot", "CIP_Title",
         "credential_category", "program_format", "is_higher_ed_provider", "is_undergrad_certificate",
         "tuition_capped", "total_contact_hours", "length_weeks",
         "completion_rate", "employment_q2_rate", "credential_rate", "median_earnings_annual",
         "has_outcomes", "last_updated"]
df = df[front + [c for c in df.columns if c not in front]]

# 1) Keep a full copy BEFORE filtering (reference only -- don't reuse it as df)
df_all = df.copy()

# 2) Certificate-only: keep certificates and industry certifications, drop everything else
#    (associate, bachelors, graduate, none, license, apprenticeship, other, missing)
df = df[df["credential_category"].isin(["certificate", "industry_certification"])]

# 2b) Catch degree programs mislabeled as certificates by checking the PROGRAM NAME
#     e.g. "Welding Technology AAS", "Bachelor of Science in Nursing", "Associate Degree in Radiologic Tech"
DEGREE_ABBR = r"\b(AAS|A\.A\.S|AA|AS|AAT|AOS|BS|BA|BAS|BAAS|BSN)\b"          # case-sensitive, so "as"/"ba" inside normal words don't match
DEGREE_PHRASES = r"associates? (of|in|degree)|bachelor|baccalaureate"    # case-insensitive

def name_says_degree(name):
    if pd.isna(name):
        return False
    s = str(name)
    low = s.lower()
    if re.search(DEGREE_ABBR, s) or re.search(DEGREE_PHRASES, low):
        return True
    # "Degree" counts only if the name doesn't also say certificate (e.g. "Post Degree Certificate" stays)
    if re.search(r"\bdegree\b", low) and not re.search(r"certific|post[- ]degree", low):
        return True
    return False

df["flag_degree_in_name"] = df["d105_program_name"].map(name_says_degree)
print("Degree programs labeled as certificates (dropped):", df["flag_degree_in_name"].sum())
df = df[~df["flag_degree_in_name"]].drop(columns="flag_degree_in_name")

print(df["credential_category"].value_counts().to_string())

# 3) Drop rows that can never be Workforce Pell
df = df[~df["is_non_us"]]                                       # outside the U.S.
df = df[df["d104_entity_type"] != "National Apprenticeship"]    # not colleges, can't receive Pell

# 4) Drop columns we no longer need
drop_cols = [
    "d133_total_wioa_served", "d134_total_wioa_exiters", "d135_total_wioa_served_with_ita",
    "d136_total_wioa_exited_with_ita", "d137_total_wioa_completed", "d138_cost_per_wioa_num",
    "d139_total_wioa_exiters_employed_q2", "d140_total_wioa_exiters_employed_q4",
    "d142_total_wioa_credential",
    "unconfirmed_de130", "unconfirmed_de170", "unconfirmed_de171",
    "d116_program_format", "d103_provider_address", "median_earnings_quarterly",
]
df = df.drop(columns=drop_cols)

# 5) Build the subsets from the FINAL df
subset = df[df["is_undergrad_certificate"] & df["is_higher_ed_provider"] & df["has_outcomes"]]
wp = df[df["likely_workforce_pell"]]

print("Before filtering:", df_all.shape)
print("Final shape:     ", df.shape)
print("Analysis subset: ", subset.shape)
print("Workforce Pell:  ", wp.shape)
print("\nMissing % in key columns:")
print((df[front].isna().mean() * 100).round(1).sort_values(ascending=False).head(12).to_string())

Degree programs labeled as certificates (dropped): 449
credential_category
certificate               18306
industry_certification    13655
Before filtering: (73604, 71)
Final shape:      (31621, 56)
Analysis subset:  (6243, 56)
Workforce Pell:   (910, 56)

Missing % in key columns:
median_earnings_annual    69.5
credential_rate           69.4
employment_q2_rate        67.3
completion_rate           60.9
length_weeks              26.3
total_contact_hours       26.3
tuition_capped             1.0
city                       0.2
zip                        0.2
provider_unique_id         0.0
program_format             0.0
has_outcomes               0.0


In [134]:
import os

output_dir = "../../../data/processed/ETPL"
os.makedirs(output_dir, exist_ok=True)

df.to_csv(f"{output_dir}/ETPL_cleaned.csv", index=False)
subset.to_csv(f"{output_dir}/ETPL_cleaned_subset.csv", index=False)
wp.to_csv(f"{output_dir}/ETPL_workforce_pell.csv", index=False)
program_text_lookup.to_csv(f"{output_dir}/ETPL_program_text_lookup.csv", index=False)

print(f"Saved 4 files to: {output_dir}")
print("Final shape:", df.shape, "| Subset:", subset.shape)

Saved 4 files to: ../../../data/processed/ETPL
Final shape: (31621, 56) | Subset: (6243, 56)
